# Analisis Keseimbangan Karakter Game Tahan Pedas

Notebook ini dibuat untuk menganalisis data hasil simulasi keseimbangan karakter dari game **Tahan Pedas** yang dimainkan oleh AI bot.

## 1. Persiapan File Data
Pastikan Anda telah mengunggah file `simulation_data.csv` ke dalam Colab Anda (seret file ke tab folder di panel kiri).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare

# Memuat data hasil simulasi
df = pd.read_csv('simulation_data.csv')
print(f"Total baris data berhasil dimuat: {len(df)}")
df.head()

## 2. Analisis Keseimbangan 2-Player (Duel)
Baseline peluang menang ideal adalah **50%**. Rentang toleransi seimbang adalah **42% - 58%**.

In [ ]:
df_2p = df[df['player_count'] == 2]

summary_2p = df_2p.groupby('character').agg(
    games_played=('game_id', 'count'),
    total_wins=('is_winner', 'sum'),
    avg_score=('score', 'mean')
).reset_index()

summary_2p['win_rate'] = (summary_2p['total_wins'] / summary_2p['games_played']) * 100
summary_2p = summary_2p.sort_values(by='win_rate', ascending=False)

print("=== Ringkasan Game 2-Player ===")
print(summary_2p.to_string(index=False))

# Visualisasi Bar Chart
plt.figure(figsize=(10, 5))
sns.barplot(data=summary_2p, x='character', y='win_rate', palette='viridis')
plt.axhline(y=50, color='red', linestyle='--', label='Ideal (50%)')
plt.axhline(y=42, color='orange', linestyle=':', label='Batas Toleransi (42% - 58%)')
plt.axhline(y=58, color='orange', linestyle=':')
plt.title('Win Rate Karakter pada Game 2-Player')
plt.ylabel('Win Rate (%)')
plt.xlabel('Karakter')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 3. Analisis Keseimbangan 3-Player
Baseline peluang menang ideal adalah **33.3%**. Rentang toleransi seimbang adalah **25.3% - 41.3%**.

In [ ]:
df_3p = df[df['player_count'] == 3]

summary_3p = df_3p.groupby('character').agg(
    games_played=('game_id', 'count'),
    total_wins=('is_winner', 'sum'),
    avg_score=('score', 'mean')
).reset_index()

summary_3p['win_rate'] = (summary_3p['total_wins'] / summary_3p['games_played']) * 100
summary_3p = summary_3p.sort_values(by='win_rate', ascending=False)

print("=== Ringkasan Game 3-Player ===")
print(summary_3p.to_string(index=False))

# Visualisasi Bar Chart
plt.figure(figsize=(10, 5))
sns.barplot(data=summary_3p, x='character', y='win_rate', palette='magma')
plt.axhline(y=33.3, color='red', linestyle='--', label='Ideal (33.3%)')
plt.axhline(y=25.3, color='orange', linestyle=':', label='Batas Toleransi (25.3% - 41.3%)')
plt.axhline(y=41.3, color='orange', linestyle=':')
plt.title('Win Rate Karakter pada Game 3-Player')
plt.ylabel('Win Rate (%)')
plt.xlabel('Karakter')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 4. Analisis Keseimbangan 4-Player
Baseline peluang menang ideal adalah **25%**. Rentang toleransi seimbang adalah **17% - 33%**.

In [ ]:
df_4p = df[df['player_count'] == 4]

summary_4p = df_4p.groupby('character').agg(
    games_played=('game_id', 'count'),
    total_wins=('is_winner', 'sum'),
    avg_score=('score', 'mean')
).reset_index()

summary_4p['win_rate'] = (summary_4p['total_wins'] / summary_4p['games_played']) * 100
summary_4p = summary_4p.sort_values(by='win_rate', ascending=False)

print("=== Ringkasan Game 4-Player ===")
print(summary_4p.to_string(index=False))

# Visualisasi Bar Chart
plt.figure(figsize=(10, 5))
sns.barplot(data=summary_4p, x='character', y='win_rate', palette='plasma')
plt.axhline(y=25.0, color='red', linestyle='--', label='Ideal (25.0%)')
plt.axhline(y=17.0, color='orange', linestyle=':', label='Batas Toleransi (17.0% - 33.0%)')
plt.axhline(y=33.0, color='orange', linestyle=':')
plt.title('Win Rate Karakter pada Game 4-Player')
plt.ylabel('Win Rate (%)')
plt.xlabel('Karakter')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 5. Karakteristik Penyebaran Skor Akhir (Boxplot)
Boxplot menampilkan distribusi perolehan skor pemain. Karakter yang konsisten (seperti Baja) cenderung memiliki kotak sempit dengan skor rendah, sementara karakter fluktuatif tinggi (seperti Rakus) akan menyebar luas.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='character', y='score', hue='player_count')
plt.title('Distribusi Skor per Karakter berdasarkan Jumlah Pemain')
plt.xlabel('Karakter')
plt.ylabel('Skor Kemenangan')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 6. Uji Signifikansi Statistik Chi-Square
Apakah perbedaan kemenangan disebabkan oleh ketidakseimbangan mekanis secara nyata atau sekadar variasi keberuntungan acak (noise)?
* Jika **p-value <= 0.05**: Ada ketidakseimbangan statistik yang signifikan.
* Jika **p-value > 0.05**: Game terbukti seimbang secara statistik (deviasi hanya kebetulan acak).

In [ ]:
for p_count in [2, 3, 4]:
    sub_df = df[df['player_count'] == p_count]
    char_wins = sub_df.groupby('character')['is_winner'].sum()
    
    total_wins = char_wins.sum()
    expected_wins = [total_wins / len(char_wins)] * len(char_wins)
    
    chi2, p_val = chisquare(f_obs=char_wins.values, f_exp=expected_wins)
    
    print(f"\n--- Uji Signifikansi untuk {p_count}-Player ---")
    print(f"Chi-Square Statistic: {chi2:.4f}")
    print(f"p-value: {p_val:.4e}")
    if p_val <= 0.05:
        print("Kesimpulan: KETIDAKSEIMBANGAN SIGNIFIKAN secara statistik (p <= 0.05). Mekanik tertentu lebih dominan.")
    else:
        print("Kesimpulan: SEIMBANG secara statistik (p > 0.05). Perbedaan kemenangan hanya akibat keberuntungan acak.")